In [1]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm


In [2]:
VALIDATION_LOOP_DIR = Path(".").resolve()
REPO_DIR = VALIDATION_LOOP_DIR.parent

sys.path.insert(0, str((REPO_DIR / "training_loop").resolve()))
sys.path.insert(0, str(REPO_DIR.resolve()))

DATASET_DIR = REPO_DIR.parent / "dataset"
TRAIN_DIR = DATASET_DIR / "Dataset_train"
VAL_DIR = DATASET_DIR / "Dataset_validation"
TRAIN_CSV = REPO_DIR / "tags" / "train.csv"
VAL_CSV = REPO_DIR / "tags" / "validation.csv"
OUTPUT_DIR = REPO_DIR / "output" / "history_3d_simple_cnn"

label_columns = ["ICH"]
target_size = 128

epoch = 11
batch_size_train = 8
batch_size_val = 8
num_workers = 4

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
from dataset_3d_binary import CTVolumeDataset


class ConvBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout_p=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.InstanceNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout3d(p=dropout_p) if dropout_p > 0 else nn.Identity(),
        )

    def forward(self, x):
        return self.block(x)


class Simple3DClassifier(nn.Module):
    def __init__(self, in_channels=1, num_classes=1):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock3D(in_channels, 16, stride=2, dropout_p=0.1),
            ConvBlock3D(16, 32, stride=2, dropout_p=0.1),
            ConvBlock3D(32, 64, stride=2, dropout_p=0.1),
            ConvBlock3D(64, 128, stride=2, dropout_p=0.1),
            ConvBlock3D(128, 256, stride=2, dropout_p=0.1),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.head(x)
        return x

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_dataset = CTVolumeDataset(
    table_path=TRAIN_CSV,
    images_dir=TRAIN_DIR,
    label_columns=label_columns,
    target_size=target_size,
)

val_dataset = CTVolumeDataset(
    table_path=VAL_CSV,
    images_dir=VAL_DIR,
    label_columns=label_columns,
    target_size=target_size,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size_train,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size_val,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
)

model = Simple3DClassifier(in_channels=1, num_classes=len(label_columns)).to(device)
checkpoint = torch.load(OUTPUT_DIR / f"checkpoint_epoch_{epoch}.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

Using device: cpu


Simple3DClassifier(
  (encoder): Sequential(
    (0): ConvBlock3D(
      (block): Sequential(
        (0): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1), bias=False)
        (1): InstanceNorm3d(16, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (2): ReLU(inplace=True)
        (3): Dropout3d(p=0.1, inplace=False)
      )
    )
    (1): ConvBlock3D(
      (block): Sequential(
        (0): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1), bias=False)
        (1): InstanceNorm3d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (2): ReLU(inplace=True)
        (3): Dropout3d(p=0.1, inplace=False)
      )
    )
    (2): ConvBlock3D(
      (block): Sequential(
        (0): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1), bias=False)
        (1): InstanceNorm3d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (2): ReLU(inplace=True)
 

In [5]:
def collect_split_predictions(model, loader, dataset, device, split_name):
    study_uids = []
    targets = []
    scores = []

    with torch.no_grad():
        for batch_idx, (volumes, labels) in enumerate(tqdm(loader, desc=f"Predict {split_name}", leave=False)):
            volumes = volumes.to(device, non_blocking=True)
            logits = model(volumes)
            probs = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)

            start = batch_idx * loader.batch_size
            end = start + len(labels)
            study_uids.extend(dataset.samples_df.iloc[start:end]["study_uid"].astype(str).tolist())
            scores.append(probs)
            targets.append(labels.numpy().reshape(-1))

    y_true = np.concatenate(targets, axis=0)
    y_score = np.concatenate(scores, axis=0)
    predictions_df = pd.DataFrame(
        {
            "study_uid": study_uids,
            "target": y_true.astype(int),
            "predict": y_score,
        }
    )
    return y_true, y_score, predictions_df


train_y_true, train_y_score, train_predictions = collect_split_predictions(
    model=model,
    loader=train_loader,
    dataset=train_dataset,
    device=device,
    split_name="train",
)

val_y_true, val_y_score, val_predictions = collect_split_predictions(
    model=model,
    loader=val_loader,
    dataset=val_dataset,
    device=device,
    split_name="validation",
)


In [8]:
train_predictions.to_csv(OUTPUT_DIR / "predict_train_3d_simple_cnn.csv", index=False)
val_predictions.to_csv(OUTPUT_DIR / "predict_val_3d_simple_cnn.csv", index=False)
